# Same-input CPU/GPU validation and timing

Benchmark both calmodulin inputs (1CFD apo and 1CLL holo-derived) with `tan` and `ionic-ass`, plus ideal and decorated periodic surfaces. Select 2048 actual contact positions per case, seed 42. CPU and GPU receive identical coordinates, descriptors and batch sizes. Use one warm-up and three synchronized repeats; record every timing, descriptor agreement, shared PCA differences and independently fitted K-means ARI.

CuPy/CUDA are optional. If GPU is unavailable or computation falls back to CPU, the result says so and no GPU speedup is reported. Radial-only timing does not predict workloads with angular CPU fallback.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image
candidates = [Path.cwd() / 'examples', Path.cwd(), Path.cwd().parent]
EXAMPLES = next((p.resolve() for p in candidates if (p / 'common.py').is_file()), None)
if EXAMPLES is None:
    raise RuntimeError('Open this notebook from the repository root or examples folder.')
sys.path[:0] = [str(EXAMPLES), str(EXAMPLES / 'periodic_surfaces'), str(EXAMPLES / 'biomapsy'), str(EXAMPLES / 'cpu_gpu')]
from common import environment, show_analysis
print(environment())


In [ ]:
from benchmark import run
report = run(points=2048, repeats=3, warmups=1, descriptor_set='radial')
folder = Path(report['output'])
print(report['gpu'])
display(Image(filename=str(folder / 'cpu_gpu_times.png')))
import pandas as pd
display(pd.read_csv(folder / 'benchmark.csv'))

## Timing scope

The plot includes CPU and verified GPU medians with observed min–max ranges. Contact construction, model setup and plotting are excluded; descriptor calculation, transfers and DataFrame assembly are included. Each NPZ stores the exact inputs and outputs. Hardware, package versions and thread settings are in `benchmark.json`.

For mixed radial/angular workloads run `run(points=512, descriptor_set="mixed")` in a separate result directory. Small jobs can favor CPU, so no universal speedup is assumed.